# Day 3 — Feature Engineering + Preprocessing

**Goal:** turn the cleaned table into numeric **X** (features) and **y** (Career labels) ready for ML.

**Today you will:**
- Build composite scores (feature engineering)
- Encode categories and scale numbers
- Split train/test **without leakage** (fit preprocessors on train only)

> **Read first:** [CONCEPTS.md](./CONCEPTS.md)


## Setup paths and imports

Run with working directory `notebooks/day03/` (Jupyter default when you open this file).

Reusable functions live in `src/preprocessing.py` — the notebook shows **how** to call them.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Walk up from notebooks/dayXX/ until we see dataset/ + requirements.txt
def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "dataset").is_dir() and (p / "requirements.txt").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found. cwd should be notebooks/dayXX/.")

PROJECT_ROOT = find_project_root()
# Allow: from src.preprocessing import ...
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    add_engineered_features,
    load_cleaned_table,
    prepare_train_test_bundle,
    save_artifacts,
    ENGINEERED_COLS,
)

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")

# Prefer Day 2 CSV; otherwise clean raw data on the fly
df = load_cleaned_table(PROJECT_ROOT)
df.shape  # (rows, columns)

## 1. Feature engineering

We **add new columns** from existing skills (see CONCEPTS for why each group makes sense).

These scores are still **inputs (X)** — we do not use `Career` inside them.

In [ ]:
engineered = add_engineered_features(df)
# Peek at the four new scores plus the target (for sanity only)
engineered[ENGINEERED_COLS + ["Career"]].head()

In [ ]:
# Summary stats — values should look reasonable (no huge outliers after Day 2 cleaning)
engineered[ENGINEERED_COLS].describe().round(2)

## 2. Train/test split + preprocessor (fit on train only)

| Step | What happens |
|------|----------------|
| Split | 80% train / 20% test, `stratify=Career` so rare careers appear in both |
| `fit` | Scaler + one-hot encoder learn **only from train** |
| `transform` | Apply the same learned mapping to train **and** test |

Output `bundle` contains numpy arrays `X_train`, `X_test`, `y_train`, `y_test`.

In [ ]:
bundle = prepare_train_test_bundle(engineered, test_size=0.2, random_state=42)
# (n_train, n_test, n_features after encoding)
bundle["n_train"], bundle["n_test"], bundle["n_features"]

In [ ]:
print("Career classes (y):", len(bundle["career_classes"]))
# First numeric names, then one-hot columns for Interest_Area
bundle["feature_names"][:6], "...", bundle["feature_names"][-2:]

## 3. Save pipeline for Day 4+ and Streamlit

Files go under `outputs/processed/` (gitignored — re-run this notebook to recreate).

In [ ]:
paths = save_artifacts(bundle, PROJECT_ROOT)
paths  # pipeline .joblib, feature_names.json, meta.json

## 4. Stratified split check

Train and test counts per career should be **roughly proportional** (not identical row counts).

In [ ]:
train_counts = pd.Series(bundle["y_train"]).value_counts().sort_index()
test_counts = pd.Series(bundle["y_test"]).value_counts().sort_index()
compare = pd.DataFrame({"train": train_counts, "test": test_counts})
# Map integer labels back to career names for readability
compare.index = [bundle["career_classes"][i] for i in compare.index]
compare.head(8)

## Checkpoint (for your report)

1. Why must `StandardScaler` be **fit** on train only?
2. Why does K-Means (Day 4) need scaled features?

**Deliverables:** `src/preprocessing.py`, saved pipeline, `X_train` / `X_test` / `y_train` / `y_test`.